# 03 — Dataset overview

For each of the 9 discovery targets: how many complexes ran, the actives-to-decoys ratio, and the protein-level constants (charge, active-site size) that stay fixed within a target. This is the Act 1 sanity check. Every later NB assumes the counts here are what you get after the 4A5S recovery.

See `docs/GLOSSARY.md` for term definitions.

_(Notebook auto-generated by `reproduce/split_monolith.py`. Self-contained: loads its data via `discovery9.io`, exports figures to `figures/03_dataset_overview_figK.png`.)_


> **Reader guide — where this notebook sits in the study.**
>
> **Part of:** *Orientation section* — no experiment.
>
> **Question this NB answers:** *what is the labelled cohort (actives vs measured non-binders,
> per target), and is it balanced enough to draw ranking conclusions?*
>
> **Method:** counts + activity histogram + per-target class balance.
>
> **Reproducibility contract:** reads `data/raw/reference/ohds_metadata.csv`; downstream
> writes go under `data/derived/02_*.csv`.
>
> **Environment:** requires `pip install -e .` from the repo root.

In [ ]:
# --- notebook preamble ---
NB_STEM = "02_dataset_overview"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')


## 1. Dataset overview

Discovery-9 is a **9-target × 30-ligand grid** — a standard NewBench slice with known actives (from ChEMBL, `pchembl` given) and property-matched decoys. Three checks up front:

1. **Completeness.** Did the analysis pipeline finish for all 270 productions?
2. **Actives/decoys balance per target.** Some targets are ~50/50; others carry more decoys. That constrains what a discrimination metric can achieve.
3. **Protein-level invariants.** Formal charge and active-site size don't change between complexes of the same target (same protein). They should be constant within a target and give a per-target baseline.


In [ ]:

aggs = {'n_complexes': ('complex_id','nunique')}
if 'is_active' in df:
    aggs['n_actives'] = ('is_active', lambda s: int((s==True).sum()))
    aggs['n_decoys']  = ('is_active', lambda s: int((s==False).sum()))
aggs['protein_q']    = ('protein_formal_charge', 'first')
aggs['as_q']         = ('active_site_formal_charge', 'first')
aggs['n_AS_res']     = ('n_active_site_residues', 'mean')

overview = df.groupby('target').agg(**aggs).round(1)
print(f'Total complexes analysed: {len(df)} / 270 expected')
overview

In [ ]:

if 'is_active' in df:
    counts = df.dropna(subset=['is_active']).groupby(['target','is_active']).size().unstack(fill_value=0)
    counts = counts.rename(columns={False:'decoys', True:'actives'})
    counts = counts.reindex(columns=[c for c in ['decoys','actives'] if c in counts.columns])
    fig, ax = plt.subplots(figsize=(9, 3.6))
    counts.plot(kind='bar', stacked=True, color=[DECOY_C, ACTIVE_C], ax=ax, edgecolor=WHITE, linewidth=0.8)
    ax.set_ylabel('# complexes'); ax.set_xlabel('')
    ax.set_title('Complexes per target — GOLD = active, NAVY = decoy')
    ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.6)
    plt.setp(ax.get_xticklabels(), rotation=0)
    ax.legend(loc='upper right', fontsize=9)

**Interpretation.** Completeness is printed above the table.

**Formal charges are per-target constants** — same protein, same pH-7 protonation state (AMBER's HID/HIE/HIP + ASH/ASP + GLH/GLU + LYN/LYS + CYM/CYS all resolved from the topology).

If `n_complexes < 30` for a target, some MD analyses timed out. If `is_active` counts are highly skewed (e.g. 3 actives / 27 decoys) the within-target actives-vs-decoys contrast gets noisy. Rule of thumb: at least 5 actives per target for a usable separation heatmap.


In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
